[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DBertazioli/aml-labs/blob/main/lab1/1_toy_pytorch.ipynb)

<img src="https://upload.wikimedia.org/wikipedia/commons/1/10/PyTorch_logo_icon.svg" alt="PyTorch" height="60">

# Lab 1, notebook 1: PyTorch basics and toy data

**Advanced Machine Learning, UNIMIB.**

Goal: learn the PyTorch workflow (tensors, autograd, `nn.Module`, the training loop) on the circles data. Then train PyTorch classifiers on four small synthetic datasets (two in 2-D, two in 3-D). See what the output activation, the loss, the width and depth, and the optimizer each change.

### Documentation

- [Learn the Basics](https://pytorch.org/tutorials/beginner/basics/intro.html) (official tutorial) and the [API reference](https://pytorch.org/docs/stable/index.html)
- [`torch.nn`](https://pytorch.org/docs/stable/nn.html) (layers and losses) and the [autograd tutorial](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)

### How to use this notebook

- Run the cells from top to bottom; a CPU runtime is enough.
- Twelve cells are marked **TODO**: replace `None` or `# TODO` with your code, then run the check cell below it.
- Sections 2-11: tensors, autograd, synthetic data and PyTorch basics, then output layers, losses, model size, the optimizer and common pitfalls.
- Sections 12-13 are theory recaps for the demos we run together in class: optimizers ([optimizer race](https://dbertazioli.github.io/aml-labs/lab1/race3d/)) and depth and initialization ([going deep](https://dbertazioli.github.io/aml-labs/lab1/deep/)). They have no TODOs.
- Solutions are released after the lab.

## 1. Setup

`set_seed` fixes the random seed, so a re-run gives the same numbers on the same setup.

`device` is `"cuda"` when a GPU is available and `"cpu"` otherwise. Nothing in this notebook needs a GPU: the data and the models are small, and they stay on the CPU.

*Further reading: [PyTorch notes, Reproducibility](https://pytorch.org/docs/stable/notes/randomness.html)*

In [ ]:
import numpy as np
import torch
import torch.nn as nn

def set_seed(s):
    torch.manual_seed(s)

set_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| numpy", np.__version__, "| device:", device)

## 2. Tensors

A tensor is an n-dimensional array, like a numpy array, that can also run on a GPU and track gradients.

Its three key properties are `shape`, `dtype` and `device`.

*Further reading: [PyTorch tutorial, Tensors](https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)*

In [ ]:
a = torch.zeros(2, 3)              # shape (2, 3), dtype float32
b = torch.arange(6).reshape(2, 3)   # dtype int64
c = torch.randn(2, 3)              # standard normal
print(a.shape, a.dtype, a.device)
print(b.dtype, "->", b.float().dtype)   # convert dtype
print(torch.ones(2, 3, device=device).device)  # create directly on device

The tensor `t` below holds `0, 1, ..., 11` as `float32`, row by row: `arange(12)` makes the numbers, `reshape(3, 4)` arranges them in 3 rows of 4. The next two subsections use it.

In [ ]:
t = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print(t)

### Broadcasting

When shapes differ, size-1 (or missing) dimensions are stretched to match.

- `(3, 4) - (4,)` subtracts the vector from every row.
- `(3, 4) - (3, 1)` subtracts one number from each row.
- `keepdim=True` keeps a reduced dimension as size 1.

*Further reading: [PyTorch notes, Broadcasting semantics](https://pytorch.org/docs/stable/notes/broadcasting.html)*

In [ ]:
print(t - t.mean(dim=0))                 # (3,4) - (4,)
print(t - t.mean(dim=1, keepdim=True))   # (3,4) - (3,1)

### Standardizing columns

Standardizing a column means computing $z = (x - \mu) / \sigma$, with the column mean $\mu$ and standard deviation $\sigma$.

`t.std(dim=0)` gives $\sigma$ for every column. By default it divides by $n - 1$; `t.std(dim=0, correction=0)` divides by $n$.

*Further reading: [PyTorch docs, torch.std](https://pytorch.org/docs/stable/generated/torch.std.html)*

> **TODO 1.** Standardize the columns of `t` using broadcasting and no loops. Store the result in `z`. Either std is fine, the default (divides by n - 1) or `correction=0` (divides by n): the check accepts both.

In [ ]:
z = None  # TODO

In [ ]:
assert z is not None, "compute z"
assert z.shape == (3, 4), f"z has shape {tuple(z.shape)}, expected (3, 4)"
assert torch.allclose(z.mean(dim=0), torch.zeros(4), atol=1e-6), "each column of z must have mean 0"
assert any(torch.allclose(z.std(dim=0, correction=c), torch.ones(4), atol=1e-5) for c in (0, 1)), \
    "each column of z must have std 1 (dividing by n or by n - 1)"

### numpy interop

- `torch.from_numpy(arr)` and `tensor.numpy()` (CPU only) **share memory** with the original: changing one changes the other.
- `torch.as_tensor(arr)` does the same when it can (same dtype, CPU). The data of this notebook goes into tensors this way.
- Use `.clone()` or `.copy()` for an independent copy.
- A tensor that requires grad or lives on a GPU needs `.detach().cpu().numpy()`.

Below, `s` is made from `arr` and doubled **in place** (`s *= 2`). The numpy array changes too.

*Further reading: [PyTorch docs, torch.from_numpy](https://pytorch.org/docs/stable/generated/torch.from_numpy.html)*

In [ ]:
arr = np.array([1., 2., 3.])
s = torch.from_numpy(arr)
s *= 2
print(arr)

## 3. Autograd

If a tensor has `requires_grad=True`, PyTorch records the operations on it.

Calling `.backward()` on a scalar fills the `.grad` of every leaf tensor with the derivative of that scalar.

*Further reading: [PyTorch tutorial, Automatic differentiation with torch.autograd](https://pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)*

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x          # dy/dx = 2x + 3 = 7 at x = 2
y.backward()
print(x.grad)

For $f(x) = x^3 + 2x$ the derivative is $f'(x) = 3x^2 + 2$, which is 14 at $x = 2$. Autograd finds the same number.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
f = x**3 + 2 * x
f.backward()
print("autograd:", x.grad.item(), "| by hand, 3x^2 + 2:", 3 * 2.0**2 + 2)

### Gradients accumulate

Each `.backward()` *adds* to `.grad`. In a training loop you must reset the gradients at every step with `optimizer.zero_grad()`.

Otherwise the step uses the sum of all past gradients.

*Further reading: [PyTorch docs, Tensor.backward](https://pytorch.org/docs/stable/generated/torch.Tensor.backward.html), [Optimizer.zero_grad](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)*

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    l = 3 * w          # d l / d w = 3 every time
    l.backward()
    print(f"step {i}: w.grad = {w.grad.item()}")   # 3, 6, 9: wrong without zero_grad

> **TODO 2.** Redo the loop above, but reset the gradient at each step so that `w.grad` is always 3.
> For a single tensor, reset it with `w.grad = None`: `.grad` starts as `None` and `backward()` then creates it. For a model with an optimizer, `optimizer.zero_grad()` does the same by default (`set_to_none=True`).

*Further reading: [PyTorch docs, Optimizer.zero_grad](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)*

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
seen = []
for i in range(3):
    # TODO: reset the gradient here
    l = 3 * w
    l.backward()
    seen.append(w.grad.item())

In [ ]:
assert seen == [3.0, 3.0, 3.0], f"w.grad was {seen} at the three steps, it must be 3 every time"

## 4. Synthetic data

The notebook uses four synthetic datasets, made by numpy generators with a fixed seed:

- `circles(n)`: class 0 is a disk, class 1 a ring around it (Gaussian noise, sd 0.20). Sections 4 to 7 use it.
- `two_spirals(n)`: two arms winding around each other (noise sd 0.085).
- `ring_chain(n)`: 3-D, three rings linked like a chain (3 classes, noise sd 0.08). The middle ring passes through the holes of the other two.
- `spiral_arms_3d(n)`: 3-D, three spiral arms rising along the z axis (3 classes, noise sd 0.05).

![The four datasets](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/data_overview.png)

*The four datasets. Colours are the classes, the same colours as in the plots below.*

Conventions:

- `n` is the number of points **per class**.
- `split` keeps 80% of the points for training and 20% for test.
- Labels `y` are integers (`0`, `1`, `2`).

The cell below defines the generators, `split`, the plot helpers and `LivePlot`, the plot that updates during training. You do not need to read it.

In [ ]:
import numpy as np

def _shuffle(X, y, rng):
    p = rng.permutation(len(y))
    return X[p].astype("float32"), y[p].astype("int64")

def circles(n, noise=0.20, seed=1):
    rng = np.random.default_rng(seed)
    r = np.r_[0.5 * np.sqrt(rng.random(n)), np.ones(n)]
    t = 2 * np.pi * rng.random(2 * n)
    X = np.c_[r * np.cos(t), r * np.sin(t)] + noise * rng.standard_normal((2 * n, 2))
    return _shuffle(X, np.r_[np.zeros(n), np.ones(n)], rng)

def two_spirals(n, noise=0.085, seed=4):
    rng = np.random.default_rng(seed)
    k = np.r_[np.zeros(n), np.ones(n)]
    r = 0.1 + 0.9 * rng.random(2 * n)
    t = r * 3 * np.pi + k * np.pi
    X = np.c_[r * np.cos(t), r * np.sin(t)] + noise * rng.standard_normal((2 * n, 2))
    return _shuffle(X, k, rng)

def _ring(rng, m, c, e1, e2, R, noise):
    t = 2 * np.pi * rng.random(m)
    return (np.array(c) + R * (np.cos(t)[:, None] * e1 + np.sin(t)[:, None] * e2)
            + noise * rng.standard_normal((m, 3)))

def ring_chain(n, noise=0.08, seed=12):
    rng = np.random.default_rng(seed)
    X, Y, Z = np.eye(3)
    rings = [_ring(rng, n, [0, 0, 0], X, Y, 1, noise),        # class 0
             _ring(rng, n, [1.3, 0, 0], X, Z, 1.3, noise),    # class 1 passes through both
             _ring(rng, n, [2.6, 0, 0], X, Y, 1, noise)]      # class 2
    return _shuffle(np.vstack(rings), np.repeat([0, 1, 2], n), rng)

def spiral_arms_3d(n, noise=0.05, seed=11):
    rng = np.random.default_rng(seed)
    k = np.repeat([0, 1, 2], n)
    h = rng.random(3 * n)
    r, t = 0.2 + 0.8 * h, k * 2 * np.pi / 3 + 5 * h
    X = np.c_[r * np.cos(t), r * np.sin(t), 2 * h - 1] + noise * rng.standard_normal((3 * n, 3))
    return _shuffle(X, k, rng)

def split(X, y, test=0.2, seed=0):
    p = np.random.default_rng(seed).permutation(len(y))
    m = int(len(y) * test)
    return X[p[m:]], y[p[m:]], X[p[:m]], y[p[:m]]


import matplotlib.pyplot as plt
import plotly.graph_objects as go

COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c"]

def to_labels(p):
    """probabilities (n, 1) or (n, K) -> integer labels"""
    return (p[:, 0] > 0.5).astype(int) if p.shape[1] == 1 else p.argmax(1)

def plot_data(X, y, title=""):
    plt.figure(figsize=(4, 4))
    plt.scatter(X[:, 0], X[:, 1], c=[COLORS[k] for k in y], s=8)
    plt.gca().set_aspect("equal"); plt.title(title); plt.show()

def plot_boundary(model, X, y, title=""):
    lo, hi = X.min(0) - 0.2, X.max(0) + 0.2
    xx, yy = np.meshgrid(np.linspace(lo[0], hi[0], 150), np.linspace(lo[1], hi[1], 150))
    grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
    z = to_labels(probs(model, grid)).reshape(xx.shape)
    plt.figure(figsize=(4, 4))
    plt.contourf(xx, yy, z, levels=[-0.5, 0.5, 1.5], colors=COLORS[:2], alpha=0.25)
    plt.scatter(X[:, 0], X[:, 1], c=[COLORS[k] for k in y], s=8)
    plt.gca().set_aspect("equal"); plt.title(title); plt.show()

def _scene(fig, title):
    fig.update_layout(title=title, height=450, margin=dict(l=0, r=0, t=40, b=0),
                      scene=dict(aspectmode="data"))
    fig.show()

def plot3d(X, y, title=""):
    fig = go.Figure([go.Scatter3d(x=X[y == k, 0], y=X[y == k, 1], z=X[y == k, 2], mode="markers",
                                  marker=dict(size=2.5, color=COLORS[k]), name=f"class {k}")
                     for k in np.unique(y)])
    _scene(fig, title)

def plot3d_pred(X, y, p, title="Test points coloured by predicted class (x = error)"):
    pred = to_labels(p); bad = pred != y
    fig = go.Figure([go.Scatter3d(x=X[pred == k, 0], y=X[pred == k, 1], z=X[pred == k, 2], mode="markers",
                                  marker=dict(size=3, color=COLORS[k]), name=f"predicted {k}")
                     for k in np.unique(y)])
    fig.add_trace(go.Scatter3d(x=X[bad, 0], y=X[bad, 1], z=X[bad, 2], mode="markers",
                               marker=dict(size=5, color="black", symbol="x"), name="errors"))
    _scene(fig, title + f": {bad.sum()} errors of {len(y)}")

import time
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

class LivePlot:
    """Keeps the per-epoch values and redraws the curves while training runs.
    update(logs) adds one epoch (redraw at most every `every` seconds); draw() shows the current state."""
    def __init__(self, every=0.5):
        self.h, self.every, self.t, self.title = {}, every, time.time(), ""   # h: (run, name) -> values per epoch
    def update(self, logs, run=""):                   # run: curve name when one plot shows several runs
        for k, v in logs.items():
            self.h.setdefault((run, k), []).append(float(v))
        n = len(self.h[(run, k)])
        self.title = f"{run + ': ' if run else ''}epoch {n} | " + " | ".join(f"{k} {float(v):.3f}" for k, v in logs.items())
        if time.time() - self.t > self.every:
            self.draw()
    def draw(self):
        if not self.h:
            return
        panels = [p for p in ("loss", "accuracy") if any(p in k for _, k in self.h)]
        several = len({run for run, _ in self.h}) > 1   # legend outside the axes when several runs are drawn
        fig, axes = plt.subplots(1, len(panels), figsize=(4.5 * len(panels) + 1.7 * several, 3), squeeze=False)
        for ax, p in zip(axes[0], panels):
            for (run, k), v in self.h.items():
                if p in k:
                    ax.plot(range(1, len(v) + 1), v, label=run or k)
            ax.set_xlabel("epoch"); ax.set_title(p)
            if several:
                ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
            else:
                ax.legend()
        fig.suptitle(self.title); fig.tight_layout()
        clear_output(wait=True); display(fig); plt.close(fig)
        self.t = time.time()

### Data as tensors

A PyTorch model takes tensors:

- Inputs: a `float32` tensor of shape `(n, d)`, one row per point. Here `d = 2`.
- `torch.as_tensor(arr)` turns a numpy array into a tensor. It shares memory with the array and keeps its dtype, which is why the generators return `float32`.
- Labels: the generators return integers, shape `(n,)`. `nn.BCELoss` wants float targets with the same shape as the model output, `(n, 1)`: `y.float().reshape(-1, 1)`. For K classes, `nn.CrossEntropyLoss` wants integer labels of shape `(n,)`, as they are.
- No preprocessing is needed here: the coordinates are roughly between -2 and 2.

For bigger data, `torch.utils.data.DataLoader` cuts the data into minibatches. Ours is small, so we do the mini-batching by hand (see Training in section 5).

*Further reading: [PyTorch tutorial, Datasets and DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html), [PyTorch docs, torch.as_tensor](https://pytorch.org/docs/stable/generated/torch.as_tensor.html)*

The data below are the circles: 500 points per class, 800 for training and 200 for test.

In [ ]:
X, y = circles(500)
X_tr, y_tr, X_te, y_te = split(X, y)                        # numpy arrays
Xt_tr, Xt_te = torch.as_tensor(X_tr), torch.as_tensor(X_te)   # float32 tensors, shape (n, 2)
yt_tr = torch.as_tensor(y_tr).float().reshape(-1, 1)          # targets for BCELoss: float, shape (n, 1)
yt_te = torch.as_tensor(y_te).float().reshape(-1, 1)
assert Xt_tr.dtype == torch.float32 and Xt_tr.shape == (800, 2)
assert yt_tr.shape == (800, 1) and set(np.unique(y_tr)) == {0, 1}
print("inputs", tuple(Xt_tr.shape), Xt_tr.dtype, "| targets", tuple(yt_tr.shape), yt_tr.dtype)
print("coordinate range:", X_tr.min().round(2), "to", X_tr.max().round(2), "| share of class 1:", y_tr.mean())
plot_data(X_tr, y_tr, "circles: training data")

## 5. PyTorch basics

Build, train and evaluate a small model on the circles data.

### Building a model

A PyTorch model is a stack of layers (`nn.Module` objects). Ours takes the 2 coordinates of a point and returns one number: the probability that the point is in class 1.

- `nn.Sequential(...)` applies the layers one after the other.
- `nn.Linear(n_in, n_out)` computes $Wx + b$. Its weight $W$ has shape `(n_out, n_in)` and its bias `(n_out,)`.
- Unlike Keras, you give the input size of every layer yourself: each layer's `n_in` must equal the previous layer's `n_out`.
- Activations are layers of their own: `nn.ReLU()`, `nn.Tanh()`, `nn.Sigmoid()`.
- A stack of Linear layers without activations is still a single linear map, so hidden layers need a non-linear activation.
- A loss is also a module: `loss(output, target)` returns the scalar to minimize.

![The demo model as a network](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/torch_model.png)

*The demo model: 2 inputs, 8 hidden ReLU units, 1 sigmoid output. Linear takes the input and output sizes; ReLU and Sigmoid are layers of their own.*

The activations as formulas:

- `nn.ReLU()`: $\max(0, z)$
- `nn.Tanh()`: $\tanh z$
- `nn.Sigmoid()`: $1/(1+e^{-z})$, output in (0, 1)

![ReLU, tanh and sigmoid](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/activations.png)

*ReLU, tanh and sigmoid for z between -4 and 4.*

*Further reading: [PyTorch tutorial, Build the Neural Network](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html), [PyTorch docs, nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)*

### Output layer and loss for two classes

**Output layer.** One output unit gives the probability $p$ that the label is 1. The **sigmoid** $1/(1+e^{-z})$ maps any real score $z$ to (0, 1), so it is the right activation for that unit.

**Loss.** Binary cross-entropy, averaged over the samples:

$$-\big(y \log p + (1 - y) \log(1 - p)\big)$$

It is close to 0 when $p$ is near the true label. It grows without bound when the model is confidently wrong.

Accuracy is not used as a loss because it is flat almost everywhere: its gradient is 0.

*Further reading: [Deep Learning book, section 8.1.2](https://www.deeplearningbook.org/contents/optimization.html) (surrogate loss functions)*

> **TODO 3.** Choose the output activation and the loss for binary classification. The check cell below passes when it runs without an error.
>
> Give them as PyTorch modules: the activation is a layer (for example `nn.ReLU()`), the loss is a module whose name contains `BCE`. The model outputs probabilities, so the loss takes probabilities (section 11 shows the other design).

In [ ]:
out_act = None  # TODO: activation module for the output unit
loss = None     # TODO: loss module

In [ ]:
assert isinstance(out_act, nn.Sigmoid), "binary output: use the sigmoid activation"
assert isinstance(loss, nn.BCELoss), "use binary cross-entropy on probabilities (BCELoss)"

The cell below builds the demo model with your output layer. `model.parameters()` yields the weight and bias tensors, and `p.numel()` is the number of elements of a tensor.

A `Linear` layer with $n_{in}$ inputs and $n_{out}$ outputs has $n_{in} n_{out}$ weights and $n_{out}$ biases. Here the first layer has 2 × 8 + 8 = 24 parameters, the second 8 + 1 = 9, total 33. The activation layers have none.

In [ ]:
set_seed(0)
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    out_act,
)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

> **TODO 4.** The model `mlp` below has two hidden layers of 16 ReLU units. Count its parameters by hand (weights plus biases of each layer) in `n_hand`, and with a one-liner over `mlp.parameters()` in `n_auto`. The two numbers must match.

*Further reading: [PyTorch docs, torch.numel](https://pytorch.org/docs/stable/generated/torch.numel.html)*

In [ ]:
mlp = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1), out_act,
)
n_hand = None  # TODO: weights + biases of each layer, written as a sum
n_auto = None  # TODO: sum of p.numel() over mlp.parameters()

In [ ]:
assert n_auto == sum(p.numel() for p in mlp.parameters()), "n_auto is not the number of elements of mlp.parameters()"
assert n_hand == n_auto, f"by hand {n_hand}, but PyTorch counts {n_auto}"

### Inside the model

`model.parameters()` yields the parameters; `model.named_parameters()` also gives their names (`"2.weight"` is the weight of the layer at index 2).

`nn.Linear` draws its weights **and** its biases uniformly from $[-b, b]$, where $n_{in}$ is the layer's input size:

$$b = \frac{1}{\sqrt{n_{in}}}$$

(This is Kaiming-uniform with `a=sqrt(5)`.) The biases are random too: they do not start at 0.

A uniform distribution on $[-b, b]$ has standard deviation $b / \sqrt{3}$. The cell below compares this with the measured value for the second layer of `mlp` (16 × 16 weights).

*Further reading: [PyTorch docs, nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) (default initialization), [He et al., Delving deep into rectifiers (2015)](https://arxiv.org/abs/1502.01852) (Kaiming initialization)*

In [ ]:
for name, p in mlp.named_parameters():
    print(f"{name:9s}", tuple(p.shape))
layer = mlp[2]                              # the second Linear layer
bound = layer.in_features ** -0.5
print("second layer: weight std:", layer.weight.std().item(), "| uniform std = bound/sqrt(3) =", bound / 3 ** 0.5)
print("second layer: all biases within the bound:", bool((layer.bias.abs() <= bound).all()), "| all zero:", bool((layer.bias == 0).all()))

### Training

PyTorch has no `fit`: you write the training loop. One pass over the training data is an **epoch**.

![The training loop of the helper fit](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/torch_loop.png)

*The loop inside the helper `fit` (defined below), with the circles numbers: 640 rows for the gradients, 160 for validation, 20 optimizer steps per epoch.*

For each epoch, the loop cuts the training data into minibatches of `batch_size` rows. For each minibatch it does four steps:

1. Forward pass: `out = model(X[idx])`.
2. Loss: `l = loss(out, y[idx])`.
3. Gradients: `optimizer.zero_grad()` clears the old ones, then `l.backward()` computes the new ones (they accumulate, see section 3).
4. `optimizer.step()` updates the weights, using the optimizer you chose.

Around it:

- `torch.randperm` shuffles the row indices at every epoch. Slices of the shuffled indices are the minibatches (manual mini-batching, no `DataLoader`).
- `model.train()` switches Dropout and BatchNorm to training mode. For evaluation use `model.eval()` and `torch.no_grad()` (section 11).
- `l.item()` turns the loss into a Python float.

The cell below runs the loop on the demo model, for 20 epochs. The plot shows the average loss of each epoch.

*Further reading: [PyTorch tutorial, Optimizing model parameters](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html), [PyTorch docs, Module.train](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.train) and [Module.eval](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval), [autograd notes, locally disabling gradient computation](https://pytorch.org/docs/stable/notes/autograd.html#locally-disabling-gradient-computation) (`torch.no_grad`)*

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
batch_size = 32
live = LivePlot()
model.train()
for epoch in range(20):
    perm = torch.randperm(len(Xt_tr))            # new shuffle every epoch
    total = 0.0
    for i in range(0, len(Xt_tr), batch_size):
        idx = perm[i:i + batch_size]             # the rows of one minibatch
        out = model(Xt_tr[idx])                  # 1. forward pass
        l = loss(out, yt_tr[idx])                # 2. loss
        optimizer.zero_grad()                    # 3. gradients: clear the old ones ...
        l.backward()                             #    ... compute the new ones
        optimizer.step()                         # 4. update the weights
        total += l.item() * len(idx)
    live.update({"loss": total / len(Xt_tr)})
live.draw()

### Helpers used in this notebook

Writing this loop for every experiment would be long. The helper `fit` is the same loop with three additions: the accuracy, a validation split and the live plot. Every later training run uses it.

The three helpers behave like their Keras counterparts:

```python
history = fit(model, loss, optimizer, X, y, epochs=E, batch_size=B, validation_split=0.2)
test_loss, test_acc = evaluate(model, loss, X, y)
p = probs(model, A)
```

1. `fit` trains the model. `history` is a dict with `loss`, `accuracy`, `val_loss`, `val_accuracy`: one value per epoch.
2. `evaluate` returns `(loss, accuracy)` on the given data.
3. `probs` returns the output for every row of `A` as a numpy array: the probability of class 1 here. If the model has several output units it applies softmax (see section 9).

`X` and `y` are numpy arrays: the helpers convert them to tensors (and the labels to the dtype and shape the loss wants).

`validation_split=0.2` holds out the last 20% of the data. It is not used for the gradient, but loss and accuracy on it are reported after every epoch.

*Further reading: the Keras convention that `fit` copies, in [Keras docs, Model.fit](https://keras.io/api/models/model_training_apis/#fit-method) (`validation_split` takes the last samples, before shuffling); the redraw uses [IPython.display.clear_output](https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#IPython.display.clear_output)*

In [ ]:
def _prep(loss, X, y):
    X = torch.as_tensor(X)
    y = torch.as_tensor(y)
    y = y.long() if isinstance(loss, nn.CrossEntropyLoss) else y.float().reshape(-1, 1)
    return X, y

def _accuracy(out, y, loss):
    if out.shape[1] == 1:       # binary: out holds probabilities (BCELoss) or logits (BCEWithLogitsLoss)
        threshold = 0.0 if isinstance(loss, nn.BCEWithLogitsLoss) else 0.5
        return ((out > threshold).float() == y).float().mean().item()
    return (out.argmax(1) == y).float().mean().item()

def evaluate(model, loss, X, y):
    X, y = _prep(loss, X, y)
    model.eval()
    with torch.no_grad():
        out = model(X)
        return loss(out, y).item(), _accuracy(out, y, loss)

def fit(model, loss, optimizer, X, y, epochs=10, batch_size=32, validation_split=0.0, live=None, run=""):
    X, y = _prep(loss, X, y)
    n_val = int(len(X) * validation_split)
    Xv, yv, X, y = X[len(X) - n_val:], y[len(X) - n_val:], X[:len(X) - n_val], y[:len(X) - n_val]
    hist = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    live = live or LivePlot()       # pass one LivePlot (and a run name) to draw several runs in one plot
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(X))
        total, hits = 0.0, 0.0
        for i in range(0, len(X), batch_size):
            idx = perm[i:i + batch_size]
            out = model(X[idx])
            l = loss(out, y[idx])
            optimizer.zero_grad()      # clear the gradients of the previous step
            l.backward()               # gradients of the loss w.r.t. all parameters
            optimizer.step()           # update the parameters
            total += l.item() * len(idx)
            hits += _accuracy(out.detach(), y[idx], loss) * len(idx)
        hist["loss"].append(total / len(X)); hist["accuracy"].append(hits / len(X))
        if n_val:
            vl, va = evaluate(model, loss, Xv, yv)
            hist["val_loss"].append(vl); hist["val_accuracy"].append(va)
        live.update({k: v[-1] for k, v in hist.items() if v}, run=run)
    live.draw()
    return hist

def probs(model, A):
    model.eval()
    with torch.no_grad():
        out = model(torch.as_tensor(A))
        if out.shape[1] > 1:
            out = out.softmax(1)
    return out.numpy()

The cell below trains a fresh copy of the demo model with `fit`: 50 epochs, Adam with learning rate 0.01, and a plot that updates while training runs. Section 12 compares optimizers.

In [ ]:
set_seed(0)
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    out_act,
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
h = fit(model, loss, optimizer, X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2)

**History**

`fit` returns a dict (`h` above) with one list per quantity, with one value per epoch: `loss`, `accuracy`, `val_loss`, `val_accuracy`.

The training values are averages over the epoch's minibatches, while the weights are still changing. The validation values are measured once, at the end of the epoch. So in the first epochs the training curve can look worse than the validation curve.

*Further reading: [Keras FAQ, why is my training loss much higher than my testing loss?](https://keras.io/getting_started/faq/#why-is-my-training-loss-much-higher-than-my-testing-loss) (the same effect in Keras)*

In [ ]:
print(list(h))
print("last epoch:", {k: round(v[-1], 3) for k, v in h.items()})

**evaluate and probs**

- `evaluate(model, loss, X, y)` returns the loss and the accuracy.
- `probs(model, X)` returns the output of the last layer for every row: here the probability of class 1, shape `(n, 1)`.

Both run in evaluation mode, without gradient tracking. The test set was not used for training or validation.

In [ ]:
test_loss, test_acc = evaluate(model, loss, X_te, y_te)
p = probs(model, X_te)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f} | probs output shape {p.shape}")

`evaluate` computes the accuracy for us. Here you compute it yourself.

> **TODO 5.** Write `accuracy(model, X, y)`. Switch the model to evaluation mode with `model.eval()` and disable gradient tracking with `torch.no_grad()`. Predict class 1 where the probability is above 0.5, and return the fraction of correct predictions as a python float.
>
> `X` is a float32 tensor of shape `(n, 2)`. The targets `y` are floats of shape `(n, 1)`, like the model output. The result must match the `test_acc` from `evaluate`.

In [ ]:
def accuracy(model, X, y):
    # TODO
    return None

In [ ]:
acc = accuracy(model, Xt_te, yt_te)
assert acc is not None and abs(acc - test_acc) < 1e-4, f"by hand {acc} but evaluate says {test_acc}"
probe, seen = nn.Sequential(nn.Linear(2, 1), out_act), []
probe.register_forward_hook(lambda module, args, output: seen.append((module.training, torch.is_grad_enabled())))
probe.train()
accuracy(probe, Xt_te, yt_te)
assert seen == [(False, False)], "use model.eval() and torch.no_grad() inside accuracy"

**What the model learned.** The background colour is the class the model predicts at each point of a grid. The dots are the test points.

In [ ]:
plot_boundary(model, X_te, y_te, "one hidden layer of 8 units, test points")

### Custom models with nn.Module

`nn.Sequential` applies layers in one chain. For any other model, write a class that inherits from `nn.Module`:

- `__init__` creates the layers and stores them as attributes. PyTorch registers their parameters, so `model.parameters()` finds them.
- `forward(self, x)` computes the output from the input. It is ordinary Python: it can have several paths, concatenations or `if` statements.
- You call the model like a function, `model(x)`, and PyTorch calls `forward`.
- `nn.Sequential` is itself a Module.

*Further reading: [PyTorch docs, nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html), [PyTorch tutorial, Build the Neural Network](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)*

`WideDeep` below is a model with two paths, a good use of `forward`. In **wide and deep**, the input goes through two Linear layers (the deep path) and also straight to the end (the wide path). `torch.cat` joins the two, and one Linear unit gives the output.

![A wide and deep model](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/torch_wide_deep.png)

*The wide and deep model as a Module: `forward` feeds the input to two paths that meet at `torch.cat`. Shapes start with `n`, the number of rows in the batch.*

- `torch.cat([x, h], dim=1)` joins along the columns: shapes `(n, 2)` and `(n, 16)` give `(n, 18)`.
- The wide path has no layer of its own. It is the input `x`, used a second time in `forward`.

`nn.Sequential` cannot express this model: its layers form one chain, and here the input feeds two paths that merge.

*Further reading: [PyTorch docs, torch.cat](https://pytorch.org/docs/stable/generated/torch.cat.html), [Cheng et al., Wide & Deep Learning for Recommender Systems (2016)](https://arxiv.org/abs/1606.07792)*

In [ ]:
class WideDeep(nn.Module):
    def __init__(self, out_act):
        super().__init__()
        self.deep = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU())
        self.out = nn.Linear(18, 1)
        self.out_act = out_act

    def forward(self, x):
        h = self.deep(x)                         # deep path, shape (n, 16)
        merged = torch.cat([x, h], dim=1)        # the wide path is the raw input x, shape (n, 18)
        return self.out_act(self.out(merged))

set_seed(0)
wide_deep = WideDeep(out_act)
print(wide_deep)
print("parameters:", sum(p.numel() for p in wide_deep.parameters()))

The next cells train the wide and deep model and a plain MLP with the same two hidden layers, on the circles.

In [ ]:
set_seed(0)
wide_deep = WideDeep(out_act)
optimizer = torch.optim.Adam(wide_deep.parameters(), lr=0.01)
h_wd = fit(wide_deep, loss, optimizer, X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2)

In [ ]:
set_seed(0)
plain = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1), out_act,
)
optimizer = torch.optim.Adam(plain.parameters(), lr=0.01)
h_plain = fit(plain, loss, optimizer, X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2)

In [ ]:
for name, m in [("plain MLP", plain), ("wide and deep", wide_deep)]:
    print(f"{name:13s} | parameters {sum(p.numel() for p in m.parameters())} | test accuracy {evaluate(m, loss, X_te, y_te)[1]:.3f}")

The wide path adds only 2 parameters: one weight from each raw input to the output unit.

Both models fit the circles about equally well: the difference is a point or two of test accuracy. With 200 test points, one point is 0.005 of accuracy. The wide path is an example of a model that `nn.Sequential` cannot express, not an improvement on this data.

## 6. Sanity checks before real training

Two quick checks catch most bugs before a long training run. Both use the circles data from section 4.

### Loss at initialization

At initialization the weights are small and random, so the output is a probability near 0.5 for every point. The cross-entropy is then close to $\ln 2 \approx 0.69$.

A value far from that points to a bug: bad initialization, unscaled inputs or the wrong loss.

*Further reading: [Karpathy, A Recipe for Training Neural Networks (2019)](https://karpathy.github.io/2019/04/25/recipe/), step "verify loss @ init"*

> **TODO 6.** Compute `init_loss`: the mean binary cross-entropy of the untrained model's probabilities `p0` against `yt_tr`.
>
> Use the loss module `loss` from TODO 3: `loss(output, target)`.

In [ ]:
set_seed(0)
model0 = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1), out_act,
)
with torch.no_grad():
    p0 = model0(Xt_tr)   # probabilities of the untrained model, shape (n, 1)
init_loss = None  # TODO: mean binary cross-entropy of p0 against yt_tr
print("init loss:", init_loss, "| ln 2 =", round(float(np.log(2)), 3))

In [ ]:
assert init_loss is not None, "compute init_loss"
assert abs(init_loss - np.log(2)) < 0.15, f"init loss {init_loss:.3f} is far from ln 2 = {np.log(2):.3f}"

### Overfit one small batch

A correct model and optimizer drive the loss on 32 samples close to 0. If they cannot, there is a bug.

*Further reading: [Karpathy, A Recipe for Training Neural Networks (2019)](https://karpathy.github.io/2019/04/25/recipe/), step "overfit one batch"; [Deep Learning book, section 11.5](https://www.deeplearningbook.org/contents/guidelines.html) ("fit a tiny dataset")*

> **TODO 7.** Overfit one batch: write the training loop for `m1` on `xb, yb`, 300 steps on the same 32 samples. Each step is a forward pass, the loss (name it `l`), `zero_grad`, `backward` and `step`.
>
> The loop already updates the live plot. Store the loss of the last step in `final_loss`.

In [ ]:
set_seed(0)
xb, yb = Xt_tr[:32], yt_tr[:32]
m1 = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1), out_act,
)
optimizer = torch.optim.Adam(m1.parameters(), lr=0.01)
live = LivePlot()
for step in range(300):
    # TODO: forward pass, loss (named l), zero_grad, backward, step
    live.update({"loss": l.item()})
live.draw()
final_loss = None  # TODO: the loss of the last step, as a float

In [ ]:
assert final_loss is not None, "store the loss of the last step in final_loss"
st = optimizer.state.get(next(m1.parameters()), {})
assert st.get("step", 0) == 300, f"expected 300 optimizer steps (one per iteration), got {int(st.get('step', 0))}"
assert final_loss < 0.01, f"final loss {final_loss:.4f} is not close to 0"

## 7. Circles: a straight line is not enough

Same circles data as in section 4.

**Logistic regression** is a model with no hidden layer: one output unit with a sigmoid on top of a linear function of the inputs.

Its decision boundary is a straight line. No straight line separates a disk from a ring around it.

In [ ]:
set_seed(0)
model = nn.Sequential(
    nn.Linear(2, 1),
    out_act,
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
h = fit(model, loss, optimizer, X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2)
test_loss, test_acc = evaluate(model, loss, X_te, y_te)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")
plot_boundary(model, X_te, y_te, "logistic regression, test points")

Accuracy stays low: a straight line does little better than the 50% chance level of two balanced classes.

Now the model of section 5 again: **one hidden layer** with 8 ReLU units. Each unit is a linear function followed by $\max(0, z)$. The output unit combines them, so the boundary can bend.

The seed is the same, so the numbers are the same as in section 5.

In [ ]:
set_seed(0)
model = nn.Sequential(
    nn.Linear(2, 8), nn.ReLU(),
    nn.Linear(8, 1), out_act,
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
h = fit(model, loss, optimizer, X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2)
test_loss, test_acc = evaluate(model, loss, X_te, y_te)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")
plot_boundary(model, X_te, y_te, "one hidden layer, test points")

## 8. Two spirals: capacity

Same output layer and loss, harder data: the two arms wind around each other.

The first model below is small: one hidden layer of 4 units, 30 epochs. It underfits for two reasons:

- It cannot represent the boundary.
- It has few optimizer steps: 30 epochs are 600 steps (20 minibatches of 32 per epoch).

In [ ]:
X, y = two_spirals(500)
X_tr, y_tr, X_te, y_te = split(X, y)
plot_data(X_tr, y_tr, "two spirals: training data")

In [ ]:
set_seed(0)
model = nn.Sequential(
    nn.Linear(2, 4), nn.ReLU(),
    nn.Linear(4, 1), out_act,
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
h = fit(model, loss, optimizer, X_tr, y_tr, epochs=30, batch_size=32, validation_split=0.2)
test_loss, test_acc = evaluate(model, loss, X_te, y_te)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")
plot_boundary(model, X_te, y_te, "small model, test points")

### Capacity

The **capacity** of a model is set by its units per layer and its number of layers.

The noise also limits the result: the arms overlap a little, so the best possible test accuracy is about 0.95, not 1.0.

> **TODO 8.** Change `hidden` (list of hidden layer widths, for example `[16, 16]`) and `epochs` so that the **test accuracy is at least 0.88**.
>
> Training should take well under a minute. If the target is missed, try a wider or deeper network first, then more epochs.

In [ ]:
set_seed(0)
hidden = [4]   # TODO: widths of the hidden layers, e.g. [16, 16]
epochs = 30    # TODO: number of epochs

widths = [2] + hidden
model = nn.Sequential(*[m for a, b in zip(widths[:-1], widths[1:]) for m in (nn.Linear(a, b), nn.ReLU())],
                      nn.Linear(widths[-1], 1), out_act)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
h = fit(model, loss, optimizer, X_tr, y_tr, epochs=epochs, batch_size=32, validation_split=0.2)
test_loss, test_acc = evaluate(model, loss, X_te, y_te)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")
plot_boundary(model, X_te, y_te, "test points")

In [ ]:
TARGET = 0.88
assert test_acc >= TARGET, f"test accuracy {test_acc:.3f} is below {TARGET}: increase hidden and/or epochs"

## 9. Ring chain: K classes

Three classes, each one a ring in 3-D, linked like a chain. No plane separates one ring from the others.

The plot is interactive: rotate it.

In [ ]:
X, y = ring_chain(400)
X_tr, y_tr, X_te, y_te = split(X, y)
plot3d(X_tr, y_tr, "ring chain: training data")

### Output layer for K classes

The last layer has K units, one per class. **Softmax** turns the K raw scores $z_1, \dots, z_K$ into probabilities:

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

The probabilities are positive and add up to 1. The predicted class is the one with the largest probability.

### Integer labels vs one-hot

Our `y` holds integers 0, 1, 2. One-hot encoding writes the same labels as `[1,0,0]`, `[0,1,0]`, `[0,0,1]`.

Categorical cross-entropy is $-\log p_c$, where $c$ is the true class. With integer labels and with one-hot labels it is the same number. Integer labels need no conversion, so we use them.

**In PyTorch:**

- `nn.CrossEntropyLoss` takes the raw scores (**logits**) and integer labels of shape `(n,)`. It applies the softmax inside the loss, for numerical stability.
- So the model ends with a plain `nn.Linear(n_in, K)` and **no** softmax layer.
- `probs` applies the softmax when we want probabilities.
- For two classes we used `nn.Sigmoid` + `nn.BCELoss` instead, so there the model itself outputs probabilities.

*Further reading: [PyTorch docs, CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) (raw scores and integer class indices), [Deep Learning book, section 4.1](https://www.deeplearningbook.org/contents/numerical.html) (softmax and numerical stability)*

> **TODO 9.** Define the output layer for 3 classes (the hidden layer below has 16 units) and the loss. Name the loss `loss3`, so that the binary `loss` from TODO 3 stays valid.

In [ ]:
out_layer = None  # TODO: the output layer for 3 classes (its input size is the hidden width)
loss3 = None      # TODO: loss module for integer labels and raw scores

In [ ]:
assert isinstance(out_layer, nn.Linear) and out_layer.out_features == 3, "3 classes need 3 output units"
assert out_layer.in_features == 16, "the hidden layer below has 16 units"
assert isinstance(loss3, nn.CrossEntropyLoss), "integer labels and raw scores: CrossEntropyLoss"

In [ ]:
set_seed(0)
model = nn.Sequential(
    nn.Linear(3, 16), nn.ReLU(),
    nn.Linear(out_layer.in_features, out_layer.out_features),   # a fresh layer of your size: re-running this cell starts from new weights
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
h = fit(model, loss3, optimizer, X_tr, y_tr, epochs=40, batch_size=32, validation_split=0.2)
test_loss, test_acc = evaluate(model, loss3, X_te, y_te)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")
plot3d_pred(X_te, y_te, probs(model, X_te))

## 10. Spiral arms: change only the optimizer

Three arms rise along the z axis, each winding around it.

In [ ]:
X, y = spiral_arms_3d(400)
X_tr, y_tr, X_te, y_te = split(X, y)
plot3d(X_tr, y_tr, "spiral arms: training data")

The model below is fixed: one hidden layer of 16 units, 40 epochs, trained with SGD at learning rate 0.01. It reaches a test accuracy of only about 0.6.

Section 12 lists the optimizer options.

> **TODO 10.** Change **only the optimizer settings** (the marked line) so that the **test accuracy is at least 0.95**.

In [ ]:
set_seed(0)
model = nn.Sequential(
    nn.Linear(3, 16), nn.ReLU(),
    nn.Linear(16, 3),
)
loss_arms = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  # TODO: change only the optimizer settings
h = fit(model, loss_arms, optimizer, X_tr, y_tr, epochs=40, batch_size=32, validation_split=0.2)
test_loss, test_acc = evaluate(model, loss_arms, X_te, y_te)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")

In [ ]:
TARGET = 0.95
assert test_acc >= TARGET, f"test accuracy {test_acc:.3f} is below {TARGET}: change the optimizer"

In [ ]:
plot3d_pred(X_te, y_te, probs(model, X_te))

### Learning rate: SGD and Adam

The learning rate matters more than almost any other setting.

The next cell trains the same model on the spiral arms with 7 optimizer settings, 20 epochs each. One plot shows the loss and the accuracy of every setting. SGD here is plain SGD, without momentum.

The same learning rate $\eta$ means a different step size for SGD and for Adam (see "Adam's step size" in the collapsed "Advanced" subsection of section 12), so their useful ranges differ.

*Further reading: [Deep Learning book, section 11.4.1](https://www.deeplearningbook.org/contents/guidelines.html), [Bengio, Practical recommendations for gradient-based training of deep architectures (2012)](https://arxiv.org/abs/1206.5533), [Kingma & Ba, Adam: a method for stochastic optimization (2014)](https://arxiv.org/abs/1412.6980), [Deep Learning book, section 8.5](https://www.deeplearningbook.org/contents/optimization.html), [PyTorch docs, SGD](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html) and [Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)*

All runs share one `LivePlot`, and `fit` draws each run as its own curve, named by `run`.

In [ ]:
X, y = spiral_arms_3d(400)
X_tr, y_tr, X_te, y_te = split(X, y)
settings = {
    "SGD lr=0.001":   lambda p: torch.optim.SGD(p, lr=0.001),
    "SGD lr=0.01":    lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD lr=0.1":     lambda p: torch.optim.SGD(p, lr=0.1),
    "Adam lr=0.0001": lambda p: torch.optim.Adam(p, lr=0.0001),
    "Adam lr=0.001":  lambda p: torch.optim.Adam(p, lr=0.001),
    "Adam lr=0.01":   lambda p: torch.optim.Adam(p, lr=0.01),
    "Adam lr=0.1":    lambda p: torch.optim.Adam(p, lr=0.1),
}
loss_arms = nn.CrossEntropyLoss()
live = LivePlot()
rows = {}
for name, make_optimizer in settings.items():
    set_seed(0)
    m = nn.Sequential(
        nn.Linear(3, 16), nn.ReLU(),
        nn.Linear(16, 3),
    )
    hist = fit(m, loss_arms, make_optimizer(m.parameters()), X_tr, y_tr, epochs=20, batch_size=32, live=live, run=name)
    rows[name] = {"final_loss": hist["loss"][-1], "test_acc": (to_labels(probs(m, X_te)) == y_te).mean()}

import pandas as pd
df = pd.DataFrame(rows).T.round(3)
df

> **TODO 11.** Using `df`, store in `best` the name (the index label) of the setting with the highest test accuracy.
>
> Then think about this: `SGD lr=0.001` and `Adam lr=0.001` use the same learning rate but end far apart. Why do SGD and Adam need different learning-rate ranges?

In [ ]:
best = None  # TODO

In [ ]:
assert best in df.index and df.loc[best, "test_acc"] == df["test_acc"].max(), "best must be the name (index label) of a setting with the highest test_acc"

## 11. Pitfalls

Most of these mistakes raise no error: the model trains, but badly. A few raise an error that is easy to misread.

### Logits and probabilities

Our binary models end with a sigmoid, so they output probabilities, and the loss is `nn.BCELoss`. The other common design is a last layer with **no activation**. It outputs a raw score called a **logit**.

A logit is the score before the sigmoid, any real number. $\sigma(0) = 0.5$, $\sigma(2) \approx 0.88$ and $\sigma(-2) \approx 0.12$. The name comes from $\mathrm{logit}(p) = \log\frac{p}{1-p}$, the inverse of the sigmoid.

With a logit output:

- The loss must take logits: `nn.BCEWithLogitsLoss` for two classes, `nn.CrossEntropyLoss` for K classes (section 9). They apply the sigmoid (or softmax) inside the loss, which is more stable numerically than a separate layer.
- The prediction is class 1 where the logit is above 0. `torch.sigmoid(z)` gives the probability.
- `nn.BCELoss` on logits raises an error: its input must be between 0 and 1.
- `nn.BCEWithLogitsLoss` on probabilities runs, but it applies a second sigmoid. The loss can then never reach 0, and the model trains badly.

*Further reading: [PyTorch docs, BCEWithLogitsLoss](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html) (the log-sum-exp trick), [Deep Learning book, section 4.1](https://www.deeplearningbook.org/contents/numerical.html) (overflow and underflow)*

The cell below computes the loss for four points with the right and the wrong combinations. The last two lines show why the stable variant matters: a logit of 30 for a point of class 0 has a loss of exactly 30, but `sigmoid(30)` is 1.0 in float32, and `nn.BCELoss` clamps its logarithm.

In [ ]:
target = torch.tensor([[1.0], [0.0], [1.0], [0.0]])
logits = torch.tensor([[2.0], [-1.0], [0.3], [-4.0]])   # all four classified correctly by z > 0
prob = torch.sigmoid(logits)
bce, bce_logits = nn.BCELoss(), nn.BCEWithLogitsLoss()
print("probabilities, BCELoss:           ", bce(prob, target).item())
print("logits, BCEWithLogitsLoss:        ", bce_logits(logits, target).item(), "(same loss, computed from the logits)")
print("probabilities, BCEWithLogitsLoss: ", bce_logits(prob, target).item(), "(wrong: a second sigmoid)")
try:
    bce(logits, target)
except RuntimeError as e:
    print("logits, BCELoss:                   RuntimeError:", e)
z, y0 = torch.tensor([[30.0]]), torch.tensor([[0.0]])
print("logit 30, BCEWithLogitsLoss:      ", bce_logits(z, y0).item())
print("logit 30, sigmoid then BCELoss:   ", bce(torch.sigmoid(z), y0).item())

A model with a logit output, trained the right way, on the circles. `fit` and `evaluate` use threshold 0 for the accuracy when the loss is `BCEWithLogitsLoss`. The model reaches about the same accuracy as the sigmoid model of section 5.

In [ ]:
X, y = circles(500)
X_tr, y_tr, X_te, y_te = split(X, y)

set_seed(0)
logit_model = nn.Sequential(
    nn.Linear(2, 8), nn.ReLU(),
    nn.Linear(8, 1),                      # no activation: the output is a logit
)
loss_logits = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(logit_model.parameters(), lr=0.01)
h = fit(logit_model, loss_logits, optimizer, X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2)
test_loss, test_acc = evaluate(logit_model, loss_logits, X_te, y_te)
print(f"test loss {test_loss:.3f} | test accuracy {test_acc:.3f}")

### Target dtype and shape

`nn.BCELoss` and `nn.BCEWithLogitsLoss` want float targets with the same shape as the output: `(n, 1)` here. The labels from the generators are integers of shape `(n,)`. Both mismatches raise an error, and the message does not say what to change.

`nn.CrossEntropyLoss` is different: its targets are integer class indices of shape `(n,)`.

*Further reading: [PyTorch docs, BCEWithLogitsLoss](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html) (shapes of input and target)*

In [ ]:
out = torch.zeros(4, 1)                       # a model output: 4 points, one logit each
y4 = torch.tensor([1, 0, 1, 0])               # integer labels, shape (4,)
cases = {"integer, shape (4,)": y4, "float, shape (4,)": y4.float(),
         "integer, shape (4, 1)": y4.reshape(-1, 1), "float, shape (4, 1)": y4.float().reshape(-1, 1)}
for name, t in cases.items():
    try:
        print(f"{name:22s} loss", nn.BCEWithLogitsLoss()(out, t).item())
    except (RuntimeError, ValueError) as e:
        print(f"{name:22s} {type(e).__name__}: {e}")

### Gradients accumulate

`backward()` adds to `.grad`: call `optimizer.zero_grad()` at every step (section 3).

### Training mode and evaluation mode

Dropout and BatchNorm behave differently in training and at evaluation.

- `model.train()` and `model.eval()` switch the mode. A new model starts in training mode.
- Evaluate and predict in eval mode. Otherwise Dropout stays random and BatchNorm uses the statistics of the batch. Call `model.train()` again before training.
- `fit` calls `model.train()` at the start of every epoch. `evaluate` and `probs` call `model.eval()`.

Dropout sets a fraction $p$ of its inputs to 0 in training mode. In eval mode it does nothing. The main models of this notebook do not use it. Regularization is the subject of Lab 2.

*Further reading: [PyTorch docs, Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html), [Srivastava et al., Dropout (2014)](https://jmlr.org/papers/v15/srivastava14a.html)*

In [ ]:
d = nn.Dropout(0.5)
v = torch.ones(1, 8)
d.train(); print("train mode:", d(v))
d.eval();  print("eval mode: ", d(v))

> **TODO 12.** Collect 1000 outputs of `d` on `v` in **training** mode: `outs`, a tensor of shape `(1000, 8)`. Then compute `train_mean`, the mean of all its values. Predict it before you run: each output above is a mix of 0 and 2.
>
> The check passes when `train_mean` is close to the inference output, 1.0.

In [ ]:
outs = None        # TODO: 1000 training-mode outputs of d on v, shape (1000, 8)
train_mean = None  # TODO: the mean of all the values in outs

In [ ]:
assert outs is not None and train_mean is not None, "compute outs and train_mean"
assert outs.shape == (1000, 8), f"outs has shape {tuple(outs.shape)}, expected (1000, 8)"
assert (outs == 0).any(), "no value was set to 0: is d in training mode? (d.train())"
assert abs(train_mean - 1.0) < 0.05, f"the mean is {train_mean:.3f}, expected about 1.0"

In training mode, Dropout multiplies the kept values by $1/(1-p) = 2$. The average output is then the same as the inference output, so the next layer sees the same scale in training and at evaluation.

*[Srivastava et al., Dropout (2014)](https://jmlr.org/papers/v15/srivastava14a.html)*

### Gradient tracking

Every operation on a tensor that requires grad is recorded, so that `backward()` can use it. The record costs memory and time. For evaluation and prediction no gradients are needed.

- `with torch.no_grad():` turns the recording off. `evaluate` and `probs` use it.
- A tensor that is part of a graph cannot become a numpy array. Use `.detach().numpy()`.

*Further reading: [autograd notes, locally disabling gradient computation](https://pytorch.org/docs/stable/notes/autograd.html#locally-disabling-gradient-computation)*

In [ ]:
lin = nn.Linear(2, 1)
x2 = torch.ones(1, 2)
print("recorded:        requires_grad =", lin(x2).requires_grad)
with torch.no_grad():
    print("under no_grad:   requires_grad =", lin(x2).requires_grad)
try:
    lin(x2).numpy()
except RuntimeError as e:
    print("RuntimeError:", e)
print("detached:", lin(x2).detach().numpy())

### Other pitfalls

- **numpy and tensors share memory.** An in-place change of one changes the other (section 2). The tensor `Xt_tr` of section 4 shares it with its array.
- **Logging the loss.** `losses.append(l)` keeps the graph of every step alive, and memory grows. Use `l.item()`. *Further reading: [PyTorch FAQ, don't accumulate history across your training loop](https://pytorch.org/docs/stable/notes/faq.html)*
- **Reusing a model.** Calling `fit` again continues from the current weights. For a fresh experiment, build a new model and a new optimizer (an optimizer holds the parameters of one model), and call `set_seed` first.
- **Device mismatch.** Model and data must be on the same device, otherwise you get `RuntimeError: Expected all tensors to be on the same device`. Move both with `.to(device)`. Here everything stays on the CPU.
- **Shapes and dtypes.** Inputs are `float32` of shape `(n, d)`. For the targets, see "Target dtype and shape" above.
- **Not normalizing the inputs.** Features on very different scales make training harder. The toy datasets need no scaling; the data of notebook 2 has columns in large units.

## 12. Optimizer recap

### Notation

- $\theta$: all the weights.
- $g$: gradient of the minibatch loss at $\theta$.
- $\eta$: learning rate.
- $t$: step number.

Every operation acts per coordinate: $g^2$ squares each entry of $g$.

### Update rules

**SGD**

$$\theta \leftarrow \theta - \eta\, g$$

The step is proportional to the gradient.

**Momentum** ($\beta \approx 0.9$)

$$v \leftarrow \beta v + g$$

$$\theta \leftarrow \theta - \eta\, v$$

$v$ is a decaying sum of past gradients: directions that keep their sign add up, directions that flip sign cancel.

*Further reading: [Sutskever et al. (2013)](https://proceedings.mlr.press/v28/sutskever13.html)*

**Nesterov**

$$v \leftarrow \beta v + g$$

$$\theta \leftarrow \theta - \eta\, (g + \beta v)$$

Momentum with the gradient taken at the look-ahead point; it reacts earlier when the step is about to overshoot.

*Further reading: [Sutskever et al. (2013)](https://proceedings.mlr.press/v28/sutskever13.html)*

**AdaGrad**

$$G \leftarrow G + g^2$$

$$\theta \leftarrow \theta - \eta\, \frac{g}{\sqrt{G} + \epsilon}$$

$G$ only grows, so every coordinate's step keeps shrinking (faster where past gradients were large), and in long runs the steps vanish.

*Further reading: [Duchi et al. (2011)](https://jmlr.org/papers/v12/duchi11a.html)*

**RMSProp** ($\rho$)

$$s \leftarrow \rho s + (1-\rho)\, g^2$$

$$\theta \leftarrow \theta - \eta\, \frac{g}{\sqrt{s} + \epsilon}$$

Like AdaGrad with a moving average, so old gradients are forgotten and steps do not vanish.

*Further reading: [Hinton, lecture 6e slides (2012)](https://www.cs.toronto.edu/~hinton/coursera/lecture6/lec6.pdf)*

**Adam** ($\beta_1 = 0.9$, $\beta_2 = 0.999$)

$$m \leftarrow \beta_1 m + (1-\beta_1)\, g$$

$$s \leftarrow \beta_2 s + (1-\beta_2)\, g^2$$

$$\hat m = \frac{m}{1-\beta_1^t}, \qquad \hat s = \frac{s}{1-\beta_2^t}$$

$$\theta \leftarrow \theta - \eta\, \frac{\hat m}{\sqrt{\hat s} + \epsilon}$$

Momentum on the gradient plus RMSProp scaling; dividing by $1-\beta^t$ corrects the bias from starting $m$ and $s$ at 0.

*Further reading: [Kingma & Ba (2014)](https://arxiv.org/abs/1412.6980)*

### Framework defaults

| Setting | Keras 3 | PyTorch |
|---|---|---|
| SGD learning rate | 0.01 | 0.001 (recent versions) |
| Adam learning rate | 0.001 | 0.001 |
| Adam / RMSProp epsilon | 1e-7 | 1e-8 |
| RMSProp decay of the squared-gradient average | `rho = 0.9` | `alpha = 0.99` |
| RMSProp learning rate | 0.001 | 0.01 |
| AdaGrad learning rate | 0.001 | 0.01 |
| AdaGrad initial accumulator `G` | 0.1 | 0 |
| AdaGrad epsilon | 1e-7 | 1e-10 |
| Momentum form | `v = m*v - lr*g`, `theta += v` | `v = m*v + g`, `theta -= lr*v` |

The two momentum forms give the same trajectory for a constant learning rate: the Keras `v` is minus `lr` times the PyTorch `v`. They differ if you change the learning rate during training.

Always set the learning rate explicitly instead of relying on defaults.

Where epsilon goes:

- Keras RMSProp and AdaGrad put it inside the square root: $g / \sqrt{s + \epsilon}$ and $g / \sqrt{G + \epsilon}$. The formulas above show it outside, as PyTorch does.
- Keras Adam puts it outside.

*Further reading: Keras [SGD](https://keras.io/api/optimizers/sgd/), [Adagrad](https://keras.io/api/optimizers/adagrad/), [RMSprop](https://keras.io/api/optimizers/rmsprop/), [Adam](https://keras.io/api/optimizers/adam/). PyTorch [SGD](https://pytorch.org/docs/stable/generated/torch.optim.SGD.html), [Adagrad](https://pytorch.org/docs/stable/generated/torch.optim.Adagrad.html), [RMSprop](https://pytorch.org/docs/stable/generated/torch.optim.RMSprop.html), [Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html)*

### Optimizer constructors

Interactive demo: [3-D optimizer race](https://dbertazioli.github.io/aml-labs/lab1/race3d/).

The constructors for this framework:

In [ ]:
params = list(nn.Linear(2, 1).parameters())   # any parameters; an optimizer needs them at creation
optimizers = {
    "SGD":      torch.optim.SGD(params, lr=0.01),
    "Momentum": torch.optim.SGD(params, lr=0.01, momentum=0.9),
    "Nesterov": torch.optim.SGD(params, lr=0.01, momentum=0.9, nesterov=True),
    "AdaGrad":  torch.optim.Adagrad(params, lr=0.01),
    "RMSProp":  torch.optim.RMSprop(params, lr=0.01, alpha=0.99),
    "Adam":     torch.optim.Adam(params, lr=0.001, betas=(0.9, 0.999)),
}
# in a training loop: optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for name, opt in optimizers.items():
    print(f"{name:9s}", {k: v for k, v in opt.defaults.items() if k in ("lr", "momentum", "nesterov", "alpha", "betas", "eps", "initial_accumulator_value")})

### Learning-rate schedules

So far the learning rate stayed the same for the whole run. A schedule changes it during training, usually at every optimizer step:

- **At the start** the weights are random, the gradients are large and Adam's running averages are still empty. A short **warmup** raises the learning rate linearly from about 0 to its peak over the first steps, so the first updates do not throw the weights far off.
- **At the end**, near a minimum, large steps jump back and forth across it. A smaller learning rate lets the model settle, and the loss drops again.

| Schedule | Learning rate at step $t$ | Typical use |
|---|---|---|
| Constant | $\eta$ | baseline |
| Step decay | multiplied by $\gamma$ (e.g. 0.1) at fixed steps or epochs | SGD on image models |
| Exponential | $\eta\, \gamma^{t}$ | smooth version of step decay |
| Cosine | $\eta \cdot \tfrac12 \left(1 + \cos(\pi t / T)\right)$: from $\eta$ to 0 at step $T$ | common default |
| Warmup + cosine | linear rise over the first steps, then cosine | short budgets, large batches, transformers |
| Reduce on plateau | divided by a factor when the validation loss stops improving | when you do not know $T$ in advance |

With a fixed number of steps $T$, as in the competition of notebook 2, warmup + cosine is a good first choice: the peak can be higher than a constant learning rate would allow, and the last steps are small.

*Further reading: [Loshchilov & Hutter, SGDR (2017)](https://arxiv.org/abs/1608.03983) (cosine), [Goyal et al. (2017)](https://arxiv.org/abs/1706.02677) (warmup), [Smith & Topin (2018)](https://arxiv.org/abs/1708.07120) (one-cycle). Keras [learning-rate schedules](https://keras.io/api/optimizers/learning_rate_schedules/) and [ReduceLROnPlateau](https://keras.io/api/callbacks/reduce_lr_on_plateau/). PyTorch [lr_scheduler](https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate)*

The schedules in this framework, plotted over 1000 steps with a peak learning rate of 0.01:

In [ ]:
T, warm = 1000, 50
S = torch.optim.lr_scheduler
schedules = {   # each entry builds a scheduler for an optimizer
    "constant":        lambda opt: S.ConstantLR(opt, factor=1.0),
    "step decay":      lambda opt: S.MultiStepLR(opt, milestones=[300, 600], gamma=0.1),
    "exponential":     lambda opt: S.ExponentialLR(opt, gamma=0.01 ** (1 / T)),
    "cosine":          lambda opt: S.CosineAnnealingLR(opt, T_max=T),
    "warmup + cosine": lambda opt: S.SequentialLR(opt, [S.LinearLR(opt, start_factor=1e-3, total_iters=warm),
                                                        S.CosineAnnealingLR(opt, T_max=T - warm)], milestones=[warm]),
}
plt.figure(figsize=(6, 3.5))
for name, make in schedules.items():
    opt = torch.optim.SGD(nn.Linear(2, 1).parameters(), lr=0.01)
    scheduler, lrs = make(opt), []
    for t in range(T):
        lrs.append(opt.param_groups[0]["lr"])
        opt.step(); scheduler.step()
    plt.plot(lrs, label=name)
plt.xlabel("optimizer step"); plt.ylabel("learning rate"); plt.legend(); plt.show()

**How to use one in PyTorch.** Build the scheduler on the optimizer and call `scheduler.step()` right after `optimizer.step()`:

```python
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(loader))   # total optimizer steps
for epoch in range(epochs):
    for xb, yb in loader:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        scheduler.step()   # once per batch, because T_max counts batches
```

- If you call `scheduler.step()` once per epoch instead, count `T_max` in epochs.
- Any formula: `LambdaLR(optimizer, lambda t: factor)` sets the learning rate to `lr * factor` at step `t`.
- Reduce on plateau needs the validation loss: `scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=5)`, then `scheduler.step(val_loss)` at the end of every epoch.
- The current learning rate is `optimizer.param_groups[0]["lr"]`. In notebook 2, `train(..., sched=scheduler)` calls `scheduler.step()` for you.

### Advanced: curvature, zigzag and step size

Background for the optimizer demo. You do not need it for the TODOs.

#### Curvature and the condition number

Near a minimum the loss is approximately

$$\tfrac12\, (\theta-\theta^\ast)^T H\, (\theta-\theta^\ast)$$

The eigenvalues of $H$, from $\lambda_{\min}$ to $\lambda_{\max}$, are the curvatures along its principal directions. The **condition number** is $\kappa = \lambda_{\max} / \lambda_{\min}$.

Gradient descent multiplies the error along a direction with curvature $\lambda$ by $(1 - \eta\lambda)$ at every step. Consequences:

**Stability.** All factors must satisfy $|1 - \eta\lambda| < 1$, so $\eta < 2/\lambda_{\max}$. Above this limit the iterates diverge.

**Speed.** The slowest direction shrinks by $1 - \eta\lambda_{\min}$ per step. At $\eta \approx 1/\lambda_{\max}$ that is $1 - 1/\kappa$, so about $\kappa$ steps reduce the error by a factor $e$.

With $\kappa = 100$ you need about 100 steps for what a well-conditioned problem does in 1.

**Zigzag.** If $\eta > 1/\lambda_{\max}$, the factor $1 - \eta\lambda_{\max}$ is negative. Each step jumps over the steep direction (the sign flips) while making slow progress along the flat one.

You cannot fix this by lowering $\eta$ without making the flat direction slower still.

*Further reading: [Deep Learning book, sections 4.2 and 4.3.1](https://www.deeplearningbook.org/contents/numerical.html) (poor conditioning, the Hessian and the step size), [section 8.2.1](https://www.deeplearningbook.org/contents/optimization.html) (ill-conditioning in neural network training)*

![Learning rate limit](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/opt_lr_limit.png)

*Quadratic with largest curvature 25: learning rate 0.07 converges, 0.08 (= 2/25) neither converges nor diverges, 0.09 diverges.*

![Optimizers on an elongated valley](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/opt_zigzag.png)

*Elongated valley: gradient descent zigzags across the steep direction, momentum overshoots before settling, Adam goes more directly to the minimum.*

#### Momentum and per-coordinate scaling

**Momentum.**

- With a constant gradient, $v$ converges to $g/(1-\beta)$. The **effective learning rate is $\eta/(1-\beta)$**, 10 times larger for $\beta = 0.9$.
- Steady directions are sped up by about $1/(1-\beta)$; oscillating ones partly cancel.
- In the form above, the stability limit for momentum is $\eta\lambda_{\max} < 2(1+\beta)$.

*Further reading: [Deep Learning book, section 8.3](https://www.deeplearningbook.org/contents/optimization.html) (momentum)*

**AdaGrad, RMSProp and Adam.**

- They divide each coordinate's step by an estimate of that coordinate's gradient size.
- This **helps only when the badly scaled directions are the coordinate axes**: $H$ nearly diagonal, one weight with large gradients and another with small ones.
- If the elongated valley is rotated against the axes (large off-diagonal entries in $H$), rescaling the axes does not reduce $\kappa$. The race demo shows both cases.

*Further reading: [Deep Learning book, section 8.5](https://www.deeplearningbook.org/contents/optimization.html) (adaptive learning rates: AdaGrad, RMSProp, Adam)*

**Adam's step size.**

- If the gradient is steady, $\hat m \approx g$ and $\sqrt{\hat s} \approx |g|$, so each weight moves about $\eta$ per step whatever the scale of the gradient. With noisy gradients the step is smaller.
- So $\eta = 10^{-3}$ means weights change by roughly $10^{-3}$ per step. For SGD the change scales with the gradient.

![Valley along the axes and rotated](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/opt_rotated.png)

*Left: the valley follows the coordinate axes. Right: the same valley rotated by 45 degrees, where per-coordinate rescaling does not reduce the condition number.*

## 13. Depth recap

### Backprop multiplies one factor per layer

Backprop sends the gradient of the loss back through the layers. Each layer multiplies it by one factor (its Jacobian, $J_l$).

With $L$ layers, the gradient at the input is $J_1 J_2 \cdots J_L$ times the gradient at the output.

![Backward pass through L layers](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/depth_chain.png)

*Backward pass: the gradient is multiplied by one factor per layer.*

- **Factors below 1:** the product shrinks toward 0 (**vanishing gradient**), for example $0.9^{50} \approx 0.005$. The first layers get almost no gradient and learn very slowly.
- **Factors above 1:** the product grows (**exploding gradient**), for example $1.1^{50} \approx 117$. Updates become huge and training diverges.

The activation sets part of the factor:

- The derivative of the sigmoid is at most 0.25, so each sigmoid layer shrinks the gradient by at least a factor 4 before the weights are counted.
- The derivative of tanh is at most 1 (it equals 1 only at 0).
- The derivative of ReLU is 0 or 1.

### Initialization scale

The weights set the other part of the factor. If they start too small, the factors are below 1; if too large, above 1.

The initial variance is chosen so that the signal keeps about the same size from layer to layer. Let $n_\text{in}$ and $n_\text{out}$ be the number of inputs and outputs of the layer (its fan-in and fan-out).

- **Glorot / Xavier:** variance $2/(n_\text{in} + n_\text{out})$. For tanh and sigmoid.
- **He:** variance $2/n_\text{in}$. For ReLU, which sets half of the units to 0 and so needs twice the variance.

Framework defaults:

- A Keras `Dense` layer uses `glorot_uniform`.
- A PyTorch `nn.Linear` uses `kaiming_uniform_` with `a=sqrt(5)`.
- That gives weights uniform in $\pm 1/\sqrt{n_\text{in}}$, with variance $1/(3\,n_\text{in})$: smaller than He.

![Gradient size per layer](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/depth_grad_layers.png)

*Gradient size per layer in a 24-layer tanh network of width 16: plain with small init, plain with Glorot init, residual with small init.*

Experiment: 24 tanh layers of width 16. The bars show the gradient size at each layer relative to the last layer.

- Plain network with small initial weights: the first layer gets $4.8 \times 10^{-7}$ of the last layer's gradient.
- Plain network with Glorot initialization: about 0.7.
- Residual network with the same small initialization: about 4.6.

### Two fixes you will see later in the course

Here is only the idea of each, without derivations.

**Residual connections.** A residual block outputs $x + F(x)$, where $F$ is a few layers. The derivative of the output with respect to $x$ is $1 + dF/dx$, so the gradient has a direct path back through the "+" and does not need to pass through $F$.

![A residual block](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/depth_residual.png)

*A residual block: output = x + F(x), with a skip connection that copies x.*

**BatchNorm.** For each unit, BatchNorm subtracts the mean over the batch and divides by the standard deviation over the batch, so the normalized values have mean 0 and standard deviation 1. Then it scales and shifts them with learned parameters $\gamma$ and $\beta$.

After that, the mean and spread of each unit's input are set by these two learned parameters instead of by the earlier weights, which keeps the per-layer factor under control.

![BatchNorm on one unit](https://raw.githubusercontent.com/DBertazioli/aml-labs/main/lab1/fig/depth_batchnorm.png)

*BatchNorm: one unit over a batch is normalized to mean 0 and std 1, then scaled and shifted.*

Interactive demos: [signal tower](https://dbertazioli.github.io/aml-labs/lab1/deep/#tower) (signal through a deep stack of layers) and [plain vs residual](https://dbertazioli.github.io/aml-labs/lab1/deep/#playground).

### References

- Vanishing and exploding gradients: [Deep Learning book, section 8.2.4](https://www.deeplearningbook.org/contents/optimization.html) (cliffs and exploding gradients), [section 6.3.2](https://www.deeplearningbook.org/contents/mlp.html) (logistic sigmoid and hyperbolic tangent), [Pascanu, Mikolov, Bengio, On the difficulty of training recurrent neural networks (2013)](https://proceedings.mlr.press/v28/pascanu13.html)
- Glorot / Xavier initialization: [Glorot & Bengio, Understanding the difficulty of training deep feedforward neural networks (2010)](https://proceedings.mlr.press/v9/glorot10a.html)
- He initialization: [He et al., Delving deep into rectifiers (2015)](https://arxiv.org/abs/1502.01852)
- Initialization in general: [Deep Learning book, section 8.4](https://www.deeplearningbook.org/contents/optimization.html)
- Framework defaults: [Keras docs, Dense](https://keras.io/api/layers/core_layers/dense/), [PyTorch docs, nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
- Residual connections: [He et al., Deep residual learning for image recognition (2016)](https://arxiv.org/abs/1512.03385)
- BatchNorm: [Ioffe & Szegedy, Batch normalization (2015)](https://arxiv.org/abs/1502.03167), [Deep Learning book, section 8.7.1](https://www.deeplearningbook.org/contents/optimization.html)